# Forecasting US Inflation Using Macroeconomic Indicators

Multiple linear regression (OLS and Ridge) predicting US CPI (inflation)
from unemployment rate, the federal funds rate, and consumer sentiment.

**Data source:** [FRED (Federal Reserve Economic Data)](https://fred.stlouisfed.org/) —
CPIAUCSL, UNRATE, FEDFUNDS, UMCSENT, monthly, January 1990–July 2026.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score

## Load Data

In [ ]:
cpi = pd.read_csv("CPIAUCSL.csv")
unrate = pd.read_csv("UNRATE.csv")
fedfunds = pd.read_csv("FEDFUNDS.csv")
umcsent = pd.read_csv("UMCSENT.csv")

cpi.head()

## Check for Missing Values

FRED's own missing-value convention uses a `.` for unpublished
observations. We check below

In [ ]:
print("CPI missing:", cpi["CPIAUCSL"].isna().sum())
print("UNRATE missing:", unrate["UNRATE"].isna().sum())
print("FEDFUNDS missing:", fedfunds["FEDFUNDS"].isna().sum())
print("UMCSENT missing:", umcsent["UMCSENT"].isna().sum())

## Handling the Missing Values

CPIAUCSL and UNRATE are each missing exactly one observation, both on
2025-10-01 — coinciding with the US federal government shutdown that month,
which suspended BLS data collection.

A custom scikit-learn transformer (`TimeSeriesInterpolator`) is used rather
than `SimpleImputer`, since `SimpleImputer` fills with a single column-wide
statistic (mean/median) — inappropriate here, as it would ignore the local
trend and replace the missing month with the entire series' average.
Linear interpolation between the neighbouring months is the correct choice
for a single, isolated gap in an otherwise smooth time series.

In [ ]:
class TimeSeriesInterpolator(BaseEstimator, TransformerMixin):
    """
    Fills missing values in a time-ordered column via linear interpolation
    between neighbouring observations. Assumes input is already sorted by
    date; no cross-sample statistic is learned, so fit() is a no-op.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in X_copy.columns:
            X_copy[col] = X_copy[col].interpolate(method="linear")
        return X_copy

In [ ]:
interpolator = TimeSeriesInterpolator()

cpi[["CPIAUCSL"]] = interpolator.fit_transform(cpi[["CPIAUCSL"]])
unrate[["UNRATE"]] = interpolator.fit_transform(unrate[["UNRATE"]])

print("CPI missing after interpolation:", cpi["CPIAUCSL"].isna().sum())
print("UNRATE missing after interpolation:", unrate["UNRATE"].isna().sum())

## Merge Series via SQL

All four series are loaded into a local SQLite database and joined on date,
since correlation and regression require the observations aligned in a
single table.

In [ ]:
conn = sqlite3.connect("macro_data.db")

cpi.to_sql("cpi", conn, if_exists="replace", index=False)
unrate.to_sql("unrate", conn, if_exists="replace", index=False)
fedfunds.to_sql("fedfunds", conn, if_exists="replace", index=False)
umcsent.to_sql("umcsent", conn, if_exists="replace", index=False)

query = """
SELECT 
    c.observation_date AS date,
    c.CPIAUCSL,
    u.UNRATE,
    f.FEDFUNDS,
    m.UMCSENT
FROM cpi c
JOIN unrate u ON c.observation_date = u.observation_date
JOIN fedfunds f ON c.observation_date = f.observation_date
JOIN umcsent m ON c.observation_date = m.observation_date
"""

data = pd.read_sql_query(query, conn)
assert len(data) == len(cpi) == len(unrate) == len(fedfunds) == len(umcsent), \
    f"join dropped rows: {len(data)}"
data["date"] = pd.to_datetime(data["date"])
data.head()

## Visualise the Data

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

axes[0].plot(data["date"], data["CPIAUCSL"])
axes[0].set_ylabel("CPI")

axes[1].plot(data["date"], data["UNRATE"])
axes[1].set_ylabel("Unemployment")

axes[2].plot(data["date"], data["FEDFUNDS"])
axes[2].set_ylabel("Fed Funds Rate")

axes[3].plot(data["date"], data["UMCSENT"])
axes[3].set_ylabel("Sentiment")
axes[3].set_xlabel("Date")

plt.suptitle("Macroeconomic Indicators Over Time")
plt.tight_layout()
plt.show()

## Interpreting the Raw Data

CPI rises steadily and smoothly across the full period, with a visibly
sharper climb from 2021 onward. FEDFUNDS and UNRATE both track the three
major macroeconomic events in this window clearly: the 2001 downturn, the
2008 financial crisis, and the 2020 COVID-19 shock. UMCSENT is noisier
month to month but falls during each of the same periods, consistent with
lower consumer confidence during downturns.

The COVID-19 shock stands out as a genuine outlier rather than a data
error: unemployment spiked from 3.5% to 14.8% in two months, far faster
and sharper than the gradual rise seen in 2008 or 2001. This event is
retained in the dataset rather than removed. Excluding it would risk
data snooping, since the decision to remove it would be made only after
seeing that it is inconvenient for the model, rather than for a reason
identified in advance. Its presence is instead noted as a limitation:
the model is fitted across a period that includes an extreme, short-lived
shock unrelated to the ordinary relationships between these indicators,
and this likely affects both models' coefficients and error estimates.

## Transforming the Target: From CPI Level to Inflation Rate

CPI itself is a price index that trends upward over the whole period, as
seen in the earlier plot. Regressing it directly on UNRATE, FEDFUNDS, and
UMCSENT risks spurious regression: two series can appear strongly related
simply because both trend over time, not because of any genuine
relationship between them.

Inflation is properly defined as the *rate of change* in CPI, not its
level. The target is therefore transformed to year-over-year percentage
change in CPI, which removes the long-term trend and is the standard,
economically meaningful definition of inflation.

In [ ]:
data = data.sort_values("date").reset_index(drop=True)
data["inflation_yoy"] = data["CPIAUCSL"].pct_change(periods=12) * 100

data = data.dropna(subset=["inflation_yoy"]).reset_index(drop=True)

data[["date", "CPIAUCSL", "inflation_yoy"]].head(10)

## Checking for Multicollinearity

Before fitting a regression, check whether the predictors are correlated
with each other.

In [ ]:
predictors = data[["UNRATE", "FEDFUNDS", "UMCSENT"]]
corr_matrix = predictors.corr()
corr_matrix

In [ ]:
pd.plotting.scatter_matrix(predictors, figsize=(8, 8), diagonal="hist")
plt.suptitle("Pairwise Relationships Between Predictors")
plt.show()

## Interpreting the Correlation Matrix

The predictors show moderate, not severe, pairwise correlation (strongest:
UNRATE and FEDFUNDS at r = -0.48), consistent with known macroeconomic
relationships (interest rate policy responds to unemployment). This is not
severe enough to make OLS unstable on its own, but Ridge regression is still
fitted as a comparison and robustness check. If the relationship is as clean
as the correlation matrix suggests, OLS and Ridge should produce similar
coefficients; a large difference would suggest the moderate correlation
matters more than it first appears.

## Predictor Correlation with Inflation

With the target now defined as the inflation rate, check how each
predictor correlates with it directly, alongside the existing
predictor-predictor correlation matrix.

In [ ]:
target_corr = data[["inflation_yoy", "UNRATE", "FEDFUNDS", "UMCSENT"]].corr()["inflation_yoy"]
target_corr

### Lagging the Predictors

The predictors so far are contemporaneous: unemployment in month *t* alongside
inflation in month *t*. That describes co-movement, not prediction. To forecast,
each predictor is shifted forward by one month, so inflation at *t* is modelled
from indicators observed at *t − 1*.

This also reflects publication reality: `inflation_yoy` at month *t* requires the
CPI release for month *t*, which is not available at *t − 1*. A contemporaneous
model would use data that does not exist at the moment the forecast is made.

In [ ]:
h = 1
base_cols = ["UNRATE", "FEDFUNDS", "UMCSENT"]

for col in base_cols:
    data[f"{col}_lag{h}"] = data[col].shift(h)

feature_cols = [f"{col}_lag{h}" for col in base_cols]

model_data = data.dropna(subset=feature_cols + ["inflation_yoy"]).reset_index(drop=True)

# Sanity check: row t's lagged value must equal row t-1's raw value.
model_data[["date", "UNRATE", "UNRATE_lag1", "inflation_yoy"]].head()

## Train/Test Split

The data is split chronologically: the earliest 80% of observations are used
for training and the most recent 20% for testing.

A random split would be invalid here. Randomly assigning months to train and
test would place future observations in the training set, letting the model
learn from data that would not have been available at the time a forecast is
made. The split must respect the direction of time.

In [ ]:
train, test = train_test_split(model_data, test_size=0.2, shuffle=False)

X_train, y_train = train[feature_cols], train["inflation_yoy"]
X_test,  y_test  = test[feature_cols],  test["inflation_yoy"]

print(f"Train: {train['date'].min().date()} to {train['date'].max().date()}  (n = {len(train)})")
print(f"Test:  {test['date'].min().date()} to {test['date'].max().date()}  (n = {len(test)})")

## Standardising the Predictors

Ridge regression penalises the sum of squared coefficients, so it is sensitive
to the scale of the inputs. UNRATE runs between roughly 3 and 15, while UMCSENT
runs between roughly 50 and 110 — without rescaling, the penalty would fall
unevenly on the predictors purely because of their units.

All three predictors are therefore standardised to zero mean and unit variance. The
scaler is fitted on the training set only and then applied to the test set. Fitting
it on the full dataset would let the test set's mean and standard deviation
influence the training data, which is a subtle form of leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)      # transform only, no fit

print("Train means (should be ~0):", X_train_scaled.mean(axis=0).round(3))
print("Test means (need NOT be 0):", X_test_scaled.mean(axis=0).round(3))

## Ordinary Least Squares

OLS chooses the coefficients that minimise the sum of squared residuals on the
training data. Because the predictors are standardised, each coefficient reads
as: the change in inflation, in percentage points, associated with a one
standard deviation increase in that predictor, holding the others fixed.

In [ ]:
ols = LinearRegression()
ols.fit(X_train_scaled, y_train)

ols_coefs = pd.Series(ols.coef_, index=feature_cols)
print(ols_coefs.round(4))
print("Intercept:", round(ols.intercept_, 4))

## Ridge Regression

Ridge minimises the squared residuals plus a penalty on the squared size of the
coefficients, controlled by a parameter alpha. Larger alpha shrinks the
coefficients toward zero, trading a little bias for lower variance. At alpha = 0
Ridge reduces exactly to OLS.

Alpha is chosen by cross-validation on the training set. `TimeSeriesSplit` is
used rather than standard k-fold: it produces folds where the validation period
always follows the training period, so no fold is ever validated on data that
precedes what it was fitted on.

In [ ]:
alphas = np.logspace(-3, 3, 50)
ridge = RidgeCV(alphas=alphas, cv=TimeSeriesSplit(n_splits=5))
ridge.fit(X_train_scaled, y_train)

print("Selected alpha:", round(ridge.alpha_, 4))

coef_comparison = pd.DataFrame({
    "OLS":   ols.coef_,
    "Ridge": ridge.coef_
}, index=feature_cols)
print(coef_comparison.round(4))

## Naive Baseline

Inflation is highly persistent: this month's rate is close to last month's. Any
regression must be compared against that fact, otherwise a low error looks like
a success when it is only a restatement of persistence.

The baseline predicts inflation in month t as simply the observed inflation in
month t-1, using no predictors at all.

In [ ]:
baseline_pred = model_data["inflation_yoy"].shift(1).loc[test.index]

print("Baseline aligned:", baseline_pred.isna().sum(), "missing values")
baseline_pred.head()

## Evaluation

Both models and the baseline are evaluated on the held-out test set using RMSE
(in percentage points of inflation) and R squared.

R squared compares each model against a constant model that always predicts the
test set mean. A negative value therefore means the model performs worse than
that constant, which is possible on held-out data and is a meaningful result
rather than an error.

Performance is reported on the training set as well as the test set. This distinguishes two failures that the test metrics alone cannot separate: a relationship that was fitted successfully and then stopped holding, versus a relationship that was never present. Only the first is a structural break; the second is a specification that carries no information at any point in the sample.

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"model": name, "RMSE": rmse, "R2": r2_score(y_true, y_pred)}

results = pd.DataFrame([
    evaluate("OLS",      y_test, ols.predict(X_test_scaled)),
    evaluate("Ridge",    y_test, ridge.predict(X_test_scaled)),
    evaluate("Baseline", y_test, baseline_pred),
]).set_index("model")

train_results = pd.DataFrame([
    evaluate("OLS",   y_train, ols.predict(X_train_scaled)),
    evaluate("Ridge", y_train, ridge.predict(X_train_scaled)),
]).set_index("model")

print("Training set:")
print(train_results.round(4))
print()
print("Test set:")
print(results.round(4))

plt.figure(figsize=(11, 5))
plt.plot(test["date"], y_test, label="Actual", linewidth=2)
plt.plot(test["date"], ols.predict(X_test_scaled), label="OLS", alpha=0.8)
plt.plot(test["date"], ridge.predict(X_test_scaled), label="Ridge", alpha=0.8)
plt.ylabel("Inflation YoY (%)")
plt.xlabel("Date")
plt.title("Test Set: Actual vs Predicted Inflation")
plt.legend()
plt.show()

## Sensitivity Check: Error by Sub-Period

The headline test metrics above are the primary result and are not revised. This
cell decomposes the same test-set errors by sub-period, without changing the
train/test split, in order to distinguish two different possible failures:

- the model fails only during the 2020-2022 disruption, or
- the model fails across the whole test period.

Alongside RMSE, the mean signed error (prediction minus actual) is reported. A
mean error near zero indicates unbiased error; a persistently positive or negative
value indicates the model sits systematically above or below actual inflation in
that sub-period, which is evidence of a level shift rather than random noise.

Note that the pre-2020 sub-period contains few observations, so its figures are
indicative only.

In [ ]:
test_eval = pd.DataFrame({
    "date":     test["date"].values,
    "actual":   y_test.values,
    "OLS":      ols.predict(X_test_scaled),
    "Ridge":    ridge.predict(X_test_scaled),
    "Baseline": baseline_pred.values,
})

def period_label(d):
    if d < pd.Timestamp("2020-01-01"):
        return "1. Pre-2020"
    if d < pd.Timestamp("2023-01-01"):
        return "2. 2020-2022 disruption"
    return "3. 2023 onward"

test_eval["period"] = test_eval["date"].apply(period_label)

rows = []
for period, g in test_eval.groupby("period"):
    for m in ["OLS", "Ridge", "Baseline"]:
        rows.append({
            "period": period,
            "model": m,
            "n": len(g),
            "RMSE": np.sqrt(mean_squared_error(g["actual"], g[m])),
            "mean_error": (g[m] - g["actual"]).mean(),
        })

subperiod = pd.DataFrame(rows).pivot(index="period", columns="model",
                                     values=["RMSE", "mean_error"])
print(test_eval.groupby("period").size().rename("n_months"))
print()
print(subperiod.round(3))

## Do the Predictors Add Anything to Persistence?

The comparison so far is between a model using only macro indicators and a baseline using only the previous month's inflation. Persistence wins, but that was close to guaranteed by the overlapping-window construction described above, so it does not establish that the indicators are uninformative.

The sharper test holds persistence fixed and asks whether the indicators improve on it. Three specifications are fitted on the same training window and evaluated on the same test set: an AR(1) using only lagged inflation, and an augmented model adding the three lagged indicators to it. If the indicators carry short-horizon information, the augmented model should beat AR(1). If the two are indistinguishable, the indicators add nothing beyond what last month's inflation already encodes.

Training-set figures are reported alongside. The augmented model nests AR(1), so it cannot fit the training window worse; any deterioration must therefore appear out of sample, which separates a relationship that has changed from one that was never estimated.

In [ ]:
model_data["inflation_yoy_lag1"] = model_data["inflation_yoy"].shift(1)

ar_cols  = ["inflation_yoy_lag1"]
aug_cols = ["inflation_yoy_lag1"] + feature_cols

def fit_eval(cols, name, on="test"):
    tr = model_data.loc[train.index].dropna(subset=cols)
    ev = model_data.loc[test.index] if on == "test" else tr
    sc = StandardScaler().fit(tr[cols])
    m = LinearRegression().fit(sc.transform(tr[cols]), tr["inflation_yoy"])
    return evaluate(name, ev["inflation_yoy"], m.predict(sc.transform(ev[cols])))

ar_train = pd.DataFrame([
    fit_eval(ar_cols,  "AR(1) only",    on="train"),
    fit_eval(aug_cols, "AR(1) + macro", on="train"),
]).set_index("model")

ar_results = pd.DataFrame([
    fit_eval(ar_cols,  "AR(1) only"),
    fit_eval(aug_cols, "AR(1) + macro"),
    evaluate("Macro only (OLS)", y_test, ols.predict(X_test_scaled)),
]).set_index("model")

print("Training set:")
print(ar_train.round(4))
print()
print("Test set:")
print(ar_results.round(4))

## Conclusion

Both regression models fail to forecast US inflation one month ahead, and are
outperformed by a naive persistence baseline across every sub-period of the test
set. They do so despite having fitted the training window, and adding them to a
persistence model makes that model worse rather than better.

### Headline results (test set, 2019-06 to 2026-07, n = 86)

| Model | RMSE (pp) | R² |
|---|---|---|
| OLS | 2.324 | −0.107 |
| Ridge | 2.336 | −0.118 |
| Persistence baseline | 0.444 | 0.960 |

Both models return a negative R², meaning they predict the test period worse
than a constant equal to the test-set mean would. The baseline, which uses no
predictors at all and simply carries forward the previous month's inflation,
achieves roughly one fifth of their error.

### The specification is not empty: it fitted, then stopped transferring

| Model | Train RMSE | Train R² | Test RMSE | Test R² |
|---|---|---|---|---|
| OLS | 0.926 | 0.347 | 2.324 | −0.107 |
| Ridge | 0.928 | 0.345 | 2.336 | −0.118 |

Over the training window the three indicators account for roughly a third of the
variance in inflation. This is a failure of generalisation, not a specification
that was empty from the start: a relationship was estimated on 1991–2019 and did
not carry over to the test period.

The in-sample figure should not be over-read. R² computed on 340 strongly
autocorrelated monthly observations is inflated relative to what an independent
sample would give, and conventional standard errors understate uncertainty under
serial correlation, so 0.347 is evidence that the level relationship fitted the
training window rather than evidence of a genuine forecasting signal. It is
nonetheless clearly distinguishable from zero, which rules out the reading that
the models never learned anything at all.

### The failure is not confined to the COVID period

The initial reading of the prediction plot suggested the models failed because
the 2021–22 inflation surge was unforecastable from the training period. The
sub-period decomposition only partly supports this:

| Period | n | Baseline RMSE | OLS RMSE | OLS mean error |
|---|---|---|---|---|
| Pre-2020 | 7 | 0.182 | 0.457 | +0.31 |
| 2020–2022 disruption | 36 | 0.544 | 3.263 | −2.32 |
| 2023 onward | 43 | 0.377 | 1.361 | +1.02 |

The disruption period is indeed where the error is largest, so the structural
break is real. But the baseline also beats OLS in both calm sub-periods, where
no shock is available as an explanation. The models are not merely defeated by
an unforecastable event; their errors in the calm sub-periods run to several
times the baseline's, despite the relationship having fitted the training window
reasonably well.

The mean signed errors show two opposite biases: the models undershoot by more
than 2pp during 2020–2022, then overshoot by roughly 1pp for the following three
years. These partially offset in the full-period RMSE, so the headline figure
understates how systematically wrong the predictions are. A persistent one-sided
error across 43 consecutive calm months indicates that the level relationship
fitted on pre-2019 data no longer holds, independently of the pandemic.

### The predictors subtract from persistence rather than adding to it

| Model | Test RMSE | Test R² |
|---|---|---|
| AR(1) only | 0.467 | 0.955 |
| AR(1) + macro | 0.527 | 0.943 |
| Macro only (OLS) | 2.324 | −0.107 |

Holding persistence fixed and adding the three lagged indicators raises test RMSE
by roughly 13%. The indicators do not merely fail to add information beyond last
month's inflation; they degrade a forecast that already has it, because the
fitted level relationship pulls predictions toward a training-era mean of 2.35
that no longer applies. Since the augmented model nests AR(1) and so cannot fit
the training window worse, the entire deterioration is out-of-sample.

One further detail points the same way: the unfitted persistence baseline (0.444)
outperforms the fitted AR(1) (0.467). Estimating even a single coefficient on
1991–2019 data costs accuracy over the test period. Nothing estimated on the
pre-2020 sample transfers cleanly, the autoregression included.

### OLS versus Ridge

The two models are near-indistinguishable (RMSE 2.324 vs 2.336). This confirms
the prediction made earlier from the correlation matrix: with pairwise predictor
correlations no stronger than −0.48, multicollinearity was never severe enough for shrinkage to matter. The selected alpha was 14.56, and the coefficients moved by less than 0.08 in every case (UNRATE −0.266 → −0.223, FEDFUNDS 0.688 → 0.649, UMCSENT −0.529 → −0.459), so the two fits are the same model to within a mild uniform shrinkage.

### Why the baseline is so strong

The persistence baseline is a demanding benchmark here, and this should be stated
rather than treated as a fair contest the models simply lost. Year-over-year
inflation at month *t* and at month *t−1* are computed from overlapping
twelve-month windows sharing eleven months of price data, so consecutive values
are close to identical by construction. Any model attempting to beat persistence
on this target must add information beyond what is already mechanically encoded
in the previous observation.

### Interpretation

The result is consistent with the documented flattening of the Phillips curve:
the contemporaneous correlation between unemployment and inflation over
1991–2026 is −0.35, in the theoretically expected direction but weak. This is an
unconditional, contemporaneous correlation, and should not be read as a direct
test of the Phillips relationship, which is properly specified as conditional and
expectations-augmented. The weak result is suggestive of, not evidence for, that
literature.

The 2021–23 inflation episode is widely attributed to supply-chain disruption,
fiscal transfers, pandemic shifts in consumption, and energy price shocks. None
of these are captured by unemployment, the federal funds rate, or consumer
sentiment. That is a plausible account of why a relationship fitted on 1991–2019
stopped holding, but it is not something this design tests. What is tested is
narrower and firmer: over 2019–2026 these three indicators degrade a persistence
forecast rather than improving it, and the relationship that held in-sample does
not survive into the test period.

### Limitations

- **Structural break in the test window.** The chronological 80/20 split places
  COVID and the 2021–23 surge entirely in the test set. No split of this dataset
  avoids that, since the break falls near the end of the sample.
- **Publication lag is approximated.** Predictors are lagged by one calendar
  month rather than by actual release date.
- **Exploratory correlations used the full sample.** The correlation matrices
  were computed before the split, so they saw test-period data. They informed
  framing rather than model selection, but a stricter treatment would compute
  them on the training set only.
- **A single horizon and a single split.** Only h = 1 was tested, and the models
  were evaluated on one holdout period. A rolling-origin backtest across multiple
  horizons would give a more robust picture.
- **Predictors enter in levels.** UNRATE and FEDFUNDS are highly persistent over
  this window, so the fitted relationship is a level relationship against a
  growth-rate target. This is a plausible reason it fails to transfer, and
  differencing the predictors was not tested.
- **The scaler is fitted before the ridge cross-validation.** Each CV fold's
  standardisation therefore uses training data from later folds. The effect is
  negligible given how little alpha matters here, but a pipeline fitted inside
  the CV would be stricter.
- **One interpolated observation.** The October 2025 CPI and UNRATE values are
  linearly interpolated rather than observed. The interpolation uses the
  following month's value, so this single test-period observation incorporates
  information unavailable in real time; the effect is negligible at one
  observation in eighty-six, but it is the same class of look-ahead the rest of
  the design avoids.

### What would be required to do better

Beating persistence at short-horizon inflation forecasting would require
variables that carry information not already contained in recent inflation:
inflation expectations series, energy and commodity prices, supply-chain pressure
indices, or wage growth. It would also require a specification permitting
regime change, since the evidence here indicates that a relationship fitted on
1991–2019 does not transfer to the post-2020 period.